In [1]:
!pip install streamlit ortools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 74.5 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.

In [2]:
%%writefile web_vrptw.py
import streamlit as st
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp

# Đồ án cuối kỳ - Môn Optimization
# Đề tài: Vehicle Routing Problem with Time Windows (VRPTW)

def main():
    st.set_page_config(page_title="Demo VRPTW", layout="centered")
    st.title("⏰ Demo VRPTW - Đồ án Optimization")
    st.caption("Bài toán điều phối xe có thêm ràng buộc khung thời gian.")
    st.markdown("---")

    if st.button("Bắt đầu giải bài toán"):

        # Ma trận thời gian di chuyển (phút)
        time_matrix = [
            [0, 6, 9, 8, 7],
            [6, 0, 8, 3, 2],
            [9, 8, 0, 11, 10],
            [8, 3, 11, 0, 1],
            [7, 2, 10, 1, 0],
        ]

        # Cửa sổ thời gian (Start, End) cho từng node
        time_windows = [
            (0, 50),   # Node 0 (Kho)
            (7, 12),   # Node 1
            (10, 15),  # Node 2
            (16, 18),  # Node 3
            (10, 13),  # Node 4
        ]
        so_xe = 2
        kho = 0

        manager = pywrapcp.RoutingIndexManager(len(time_matrix), so_xe, kho)
        routing = pywrapcp.RoutingModel(manager)

        def time_callback(from_index, to_index):
            a = manager.IndexToNode(from_index)
            b = manager.IndexToNode(to_index)
            return time_matrix[a][b]

        transit_callback_idx = routing.RegisterTransitCallback(time_callback)
        routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_idx)

        # Thêm chiều thời gian (Time Dimension) để quản lý ràng buộc giờ giấc
        # Tham số: (callback, max_chờ, max_chạy, force_start_zero, tên)
        routing.AddDimension(transit_callback_idx, 30, 30, False, 'Time')
        time_dim = routing.GetDimensionOrDie('Time')

        # Add ràng buộc khung giờ vào từng điểm cụ thể
        for idx, (t_start, t_end) in enumerate(time_windows):
            if idx == kho:
                continue # Kho thì bỏ qua
            node_idx = manager.NodeToIndex(idx)
            time_dim.CumulVar(node_idx).SetRange(t_start, t_end)

        search_params = pywrapcp.DefaultRoutingSearchParameters()
        search_params.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC

        solution = routing.SolveWithParameters(search_params)

        if solution:
            st.success(f"Tổng thời gian hoạt động: {solution.ObjectiveValue()} phút")

            for xe in range(so_xe):
                index = routing.Start(xe)
                route_str = f"**Lộ trình xe {xe}:**\n\n"

                while not routing.IsEnd(index):
                    t_var = time_dim.CumulVar(index)
                    node = manager.IndexToNode(index)
                    # Min() là thời gian xe chạy đến nơi
                    route_str += f"📍 Node {node} (Tới lúc: {solution.Min(t_var)}p) ➡️ "
                    index = solution.Value(routing.NextVar(index))

                t_var = time_dim.CumulVar(index)
                route_str += f"🏠 Về Kho (Lúc: {solution.Min(t_var)}p)"

                st.info(route_str)
        else:
            st.error("Không xếp được lịch! Bị kẹt ràng buộc thời gian (Time Windows).")

if __name__ == '__main__':
    main()

Writing web_vrptw.py


In [3]:
!wget -q -O - ipv4.icanhazip.com
!streamlit run web_vrptw.py & npx --yes localtunnel --port 8501

35.233.239.109
⠙⠹

⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇2026-06-16 12:51:42.568 Uvicorn server started on 0.0.0.0:8501
⠏⠋
  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.233.239.109:8501

your url is: https://shaky-apples-judge.loca.lt
  Stopping...
^C
